# 09 — 外部触发器与同步

本笔记本介绍协调 Python 与 IviumSoft 的所有可用机制：

- **状态参数** — Python 和 IviumSoft 批处理脚本均可读写的共享整数
- **数字输出触发器** — Python 驱动硬件线；IviumSoft 批处理暂停直到检测到边沿
- **模拟输出触发器** — Python 将 DAC 电压升至阈值以上；IviumSoft 批处理暂停直到满足电平
- **IviumSoft → Python** — IviumSoft 批处理通过 `ExecuteProgram` 启动 Python 脚本并等待其退出
- **多实例广播** — 同时向所有活跃 IviumSoft 实例发送触发信号

### 各机制适用场景

| 机制 | 最佳场景 |
|-----------|----------|
| 状态参数 | 纯软件协调 — 无需额外接线，可通过任何连接工作 |
| 数字触发器 | 硬实时边沿：亚毫秒延迟，也适用于外部仪器 |
| 模拟触发器 | 基于阈值的启动：来自传感器或 DAC 的电压电平 |
| ExecuteProgram | IviumSoft 驱动 Python（后处理、通知、文件生成） |

---

> ⚠️ **未经生产测试。** 本笔记本中的模式源自 IviumSoft DLL 参考手册。没有任何一种模式在生产环境的真实硬件上经过验证。时序值、批处理配置细节和边缘情况行为可能与此处描述的有所不同。
>
> **需要反馈：** 如果你尝试了这些模式并发现行为有所不同 — 时序错误、触发丢失、批处理配置不正确，或任何意外情况 — 请在 https://github.com/Gnpd/pyvium/issues 提交问题，附上你的 IviumSoft 版本和观察到的现象描述。你的实际使用经验将使本笔记本更加可靠。

---

### 前提条件
- IviumSoft 已安装并运行
- 驱动已打开（参见 `01_getting_started.ipynb`）
- 数字/模拟触发器：IviumSoft 批处理方法已配置匹配的 `DirectCommand`
- 错误处理参考：`01_getting_started.ipynb` — 第 4 节

In [ ]:
import time
from pyvium import Pyvium

Pyvium.open_driver()
print(f"活跃实例: {Pyvium.get_active_iviumsoft_instances()}")

## 1. 状态参数

`set_status_par(value)` / `get_status_par()` 公开了 Python 与 IviumSoft 之间共享的全局整数。它是**每实例的** — 每个 IviumSoft 窗口维护自己独立的值。

**来自 IviumSoft 手册的关键事实：**
- 在 PC 启动时初始化为 **`-1`**
- **保留上次设置的值**直到 PC 重启 — 不会在测量或驱动打开/关闭之间重置
- Python（通过 pyvium）和 IviumSoft 批处理脚本均可读写
- DLL 函数为 `IV_setstatuspar` / `IV_getstatuspar`

### 推荐约定

| 值 | 设置方 | 含义 |
|-------|------------|------|
| `-1` | PC 启动默认 | 未初始化 |
| `0` | Python | 重置 / 空闲 |
| `1` | Python |「就绪 — IviumSoft 可以开始」 |
| `2` | IviumSoft 批处理 | 「测量运行中」 |
| `3` | IviumSoft 批处理 | 「测量完成」 |
| `-99` | 任一方 | 中止 / 错误 |

> **始终在每次工作流开始时显式重置为 `0`。** 由于值在 PC 重启前一直保留，上一个会话遗留的值可能导致 IviumSoft 批处理的 `WaitForStatusPar` 立即触发并跳过预期的等待。

In [ ]:
Pyvium.select_iviumsoft_instance(1)

# 新工作流开始前始终重置
Pyvium.set_status_par(0)
print(f"已重置为 0")

# 读回
val = Pyvium.get_status_par()
print(f"当前值: {val}")

## 2. Python → IviumSoft：告知 IviumSoft 开始

模式：Python 完成准备工作后，将状态参数设置为 `1`。已运行的 IviumSoft 批处理脚本在 `WaitForStatusPar` 行处暂停，看到 `1` 后继续执行。

**IviumSoft 批处理配置：**
```
DirectCommand:
  WaitForStatusPar  ✓   Value = 1   Timeout = 300 s
```

**流程：**
```
Python:    set_status_par(0)        ← 重置
Python:    ... 准备实验 ...
Python:    set_status_par(1)        ← 触发
IviumSoft: WaitForStatusPar 退出   ← 批处理继续
```

In [ ]:
Pyvium.select_iviumsoft_instance(1)
Pyvium.set_status_par(0)   # 重置
print("状态参数已重置为 0（空闲）")

# ... 此处放置准备工作 ...
time.sleep(0.5)            # 模拟准备时间

Pyvium.set_status_par(1)   # 触发
print("已触发 IviumSoft（状态参数 = 1）")

## 3. IviumSoft → Python：等待 IviumSoft 完成

模式：Python 在循环中轮询 `get_status_par()`，等待 IviumSoft 设置特定值表示完成。IviumSoft 批处理端使用 `SetStatusPar` DirectCommand 写入该值。

**IviumSoft 批处理配置：**
```
DirectCommand:
  SetStatusPar  ✓   Value = 3     ← 测量完成时写入
```

**流程：**
```
IviumSoft: ... 运行实验 ...
IviumSoft: SetStatusPar = 3        ← 向 Python 发出信号
Python:    get_status_par() → 3    ← 轮询循环退出
Python:    collect_data()
```

In [ ]:
def wait_for_status(instance: int, target: int, timeout_s: float = 120.0) -> bool:
    """阻塞直到指定实例的状态参数达到 target，或超时。"""
    Pyvium.select_iviumsoft_instance(instance)
    t0 = time.time()
    while True:
        current = Pyvium.get_status_par()
        if current == target:
            return True
        if time.time() - t0 > timeout_s:
            print(f"超时：实例 {instance} 未达到状态={target}")
            return False
        time.sleep(0.2)

print("wait_for_status() 已定义")

# 等待最多 60 s 直到 IviumSoft 将状态参数设为 3（「完成」）
# done = wait_for_status(instance=1, target=3, timeout_s=60.0)
# if done:
#     print("IviumSoft 完成 — 收集数据")

## 4. 完整双向协议

结合第 2 和第 3 节，产生完整握手：Python 告知 IviumSoft 开始，IviumSoft 告知 Python 完成。

| 步骤 | Python 端 | IviumSoft 批处理端 |
|------|-------------|----------------------|
| 1 | `set_status_par(0)` — 重置 | — |
| 2 | （准备） | `WaitForStatusPar`（value=1，timeout=300s）— 等待 |
| 3 | `set_status_par(1)` — 执行 | `WaitForStatusPar` 退出，批处理继续 |
| 4 | `wait_for_status(1, target=3)` — 轮询 | （运行实验）→ `SetStatusPar`（value=3） |
| 5 | 轮询退出，收集数据 | — |

> **两端均设置超时：** 在 `WaitForStatusPar`（IviumSoft 端）和 `wait_for_status()`（Python 端）中始终配置超时。任一端丢失信号都绝不能永久停滞序列。

In [ ]:
def run_with_handshake(instance: int, timeout_s: float = 120.0):
    Pyvium.select_iviumsoft_instance(instance)

    # 第 1 步：重置
    Pyvium.set_status_par(0)
    print(f"[实例 {instance}] 已重置 — 状态=0")

    # 第 2-3 步：告知 IviumSoft 开始
    Pyvium.set_status_par(1)
    print(f"[实例 {instance}] 已触发 — 状态=1")

    # 第 4 步：等待 IviumSoft 报告完成（状态=3）
    done = wait_for_status(instance, target=3, timeout_s=timeout_s)
    if done:
        print(f"[实例 {instance}] IviumSoft 完成 — 状态=3")
    return done

print("run_with_handshake() 已定义")
# run_with_handshake(instance=1, timeout_s=60.0)

## 5. 数字输出触发器

Python 驱动连接到 CompactStat 外部数字输入的数字输出线。IviumSoft 批处理在 `WaitForDigIn1` 处暂停直到检测到边沿。

**IviumSoft 批处理配置：**
```
DirectCommand:
  WaitForDigIn1  ✓   WaitForHi = ✓   Timeout = 60 s
```

**硬件连接：**
```
CompactStat 外部端口
  数字输出 0  ──────────────────────  数字输入 1
                    （短接线）
```

`set_digital_output(bitmask)` 同时控制所有数字输出线。每个位对应一条线：
```
0b00000001  →  第 0 线为高电平
0b00000100  →  第 2 线为高电平
0b00000000  →  所有线为低电平
```

> **时序：** IviumSoft 大约每 10–50 ms 轮询一次 `DigIn1`。短于约 50 ms 的脉冲可能被错过。在释放前将该线保持高电平至少 100 ms。

In [ ]:
TRIGGER_LINE = 0   # 接线到设备 DigIn1 的数字输出线
HOLD_TIME_S  = 0.1 # 保持高电平 100 ms — IviumSoft 足够检测到

def trigger_digital(line: int = TRIGGER_LINE, hold_s: float = HOLD_TIME_S):
    """将数字输出线置为高电平，保持，然后释放。"""
    Pyvium.set_digital_output(1 << line)
    print(f"数字第 {line} 线 → 高电平")
    time.sleep(hold_s)
    Pyvium.set_digital_output(0)
    print(f"数字第 {line} 线 → 低电平（已释放）")

def read_digital_inputs() -> dict:
    """以字典形式返回所有数字输入线的当前状态。"""
    bitmask = Pyvium.get_digital_input()
    return {line: bool(bitmask & (1 << line)) for line in range(8)}

print("数字触发器辅助函数已定义")

# 读取当前数字输入状态
inputs = read_digital_inputs()
for line, state in inputs.items():
    print(f"  数字输入 {line}: {'高电平' if state else '低电平'}")

# 触发
# trigger_digital()

## 6. 模拟输出触发器

Python 将 DAC 通道 0 升至阈值电压以上。IviumSoft 批处理在 `WaitForAn1` 处暂停直到模拟输入 1 超过配置的阈值。

**IviumSoft 批处理配置：**
```
DirectCommand:
  WaitForAn1  ✓   UntilAn1 > 2.5 V   Timeout = 60 s
```

**硬件连接：**
```
CompactStat 外部端口
  DAC 通道 0  ──────────────────────  模拟输入 1
                    （短接线）
```

**选择阈值：**
- 将 IviumSoft 阈值设置为明显高于空闲 DAC 电压且低于触发电压的值
- 示例：空闲 = 0 V，触发 = 3.0 V，IviumSoft 阈值 = 2.5 V
- 避免阈值接近空闲电平 — DAC 噪声可能导致误触发

> **DAC 输出范围：** 通常为 ±10 V。超过 ±5 V 前请查阅设备规格。

In [ ]:
TRIGGER_VOLTAGE = 3.0   # V — 必须超过 IviumSoft 批处理阈值
IDLE_VOLTAGE    = 0.0   # V — 不触发时的安全电平
DAC_CHANNEL     = 0     # 接线到模拟输入 1 的 DAC 通道
HOLD_TIME_S     = 0.5   # 保持在阈值以上足够长时间供 IviumSoft 检测

def trigger_analog(
    channel: int = DAC_CHANNEL,
    trigger_v: float = TRIGGER_VOLTAGE,
    idle_v: float = IDLE_VOLTAGE,
    hold_s: float = HOLD_TIME_S,
):
    """将 DAC 升至 trigger_v，保持，然后返回 idle_v。"""
    Pyvium.set_dac(channel, trigger_v)
    print(f"DAC {channel} → {trigger_v} V（触发）")
    time.sleep(hold_s)
    Pyvium.set_dac(channel, idle_v)
    print(f"DAC {channel} → {idle_v} V（空闲）")

def read_analog_inputs(n_channels: int = 4) -> dict:
    """读取前 n_channels 个 ADC 输入并以字典返回。"""
    return {ch: Pyvium.get_adc(ch) for ch in range(n_channels)}

print("模拟触发器辅助函数已定义")

# 读取当前 ADC 值
adc = read_analog_inputs()
for ch, v in adc.items():
    print(f"  ADC {ch}: {v:.4f} V")

# 触发
# trigger_analog()

## 7. IviumSoft 启动 Python（`ExecuteProgram`）

反向方向：IviumSoft 批处理脚本在其序列的特定位置启动 Python 脚本，并可选择等待其退出后再继续。

**IviumSoft 批处理配置：**
```
DirectCommand:
  ExecuteProgram     ✓
  Program Name:      python.exe
  Path:              C:\Users\...\Scripts\
  Command Line:      C:\path\to\my_script.py --output C:\data\result.csv
  WaitUntilFinished: ✓
```

**典型序列：**
```
[批处理第 N 行]    ExecuteMethod           ← 运行电化学实验
[批处理第 N+1 行]  ExecuteProgram          ← 启动 Python，等待退出
[批处理第 N+2 行]  循环返回 / 下一次扫描   ← Python 结果已在磁盘上
```

**使用场景：**
- 在批处理循环中每次扫描后立即后处理数据
- 长时间实验完成后发送通知（邮件、Slack）
- 读取测量结果并为下一个批处理行写入参数文件
- 运行质量控制 — 结果超出范围时中止批处理

**退出码约定：**
IviumSoft 不检查 Python 退出码 — 它只等待进程结束。要从 Python 回传信号给 IviumSoft，在退出前写入文件或使用 `set_status_par()`。

In [ ]:
# 由 IviumSoft 通过 ExecuteProgram 启动的脚本模板
# 保存为独立 .py 文件；IviumSoft 将调用它并等待其退出。

TEMPLATE = '''
import sys
import argparse
from pyvium import Pyvium

parser = argparse.ArgumentParser()
parser.add_argument("--output", required=True)
args = parser.parse_args()

Pyvium.open_driver()
Pyvium.select_iviumsoft_instance(1)

n = Pyvium.get_available_data_points_number()
rows = [Pyvium.get_data_point(i) for i in range(1, n + 1)]

import csv
with open(args.output, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["E (V)", "I (A)", "aux"])
    writer.writerows(rows)

# 可选：通过状态参数向 IviumSoft 回传信号
Pyvium.set_status_par(10)   # "Python 后处理完成"

Pyvium.close_driver()
sys.exit(0)
'''

print("ExecuteProgram 脚本模板:")
print(TEMPLATE)

## 8. 多实例广播

要近乎同时触发所有活跃 IviumSoft 实例，遍历它们并在每个上设置状态参数。循环本身只需几毫秒，因此所有实例在一个 IviumSoft 轮询周期内都会收到信号。

这是同时在多台设备上开始并行实验的标准模式。

In [ ]:
def broadcast_status(value: int):
    """在所有活跃 IviumSoft 实例上设置状态参数。"""
    instances = Pyvium.get_active_iviumsoft_instances()
    for inst in instances:
        Pyvium.select_iviumsoft_instance(inst)
        Pyvium.set_status_par(value)
    print(f"已向实例 {instances} 广播状态={value}")
    return instances

def wait_all_status(instances: list, target: int, timeout_s: float = 120.0) -> bool:
    """等待每个实例的状态参数等于 target，或超时。"""
    remaining = set(instances)
    t0 = time.time()
    while remaining:
        finished = set()
        for inst in remaining:
            Pyvium.select_iviumsoft_instance(inst)
            if Pyvium.get_status_par() == target:
                finished.add(inst)
                print(f"  实例 {inst}: 状态={target}")
        remaining -= finished
        if remaining and time.time() - t0 > timeout_s:
            print(f"超时 — 仍在等待实例 {remaining}")
            return False
        if remaining:
            time.sleep(0.2)
    return True

print("广播辅助函数已定义")

# 重置所有，然后触发所有
instances = broadcast_status(0)   # 重置
time.sleep(0.5)
broadcast_status(1)               # 执行

## 9. 选择触发类型

### 状态参数 vs 硬件触发器

| 标准 | 状态参数 | 数字触发器 | 模拟触发器 |
|-----------|-----------------|-----------------|----------------|
| 是否需要接线 | 否 | 是（1 根线） | 是（1 根线） |
| 延迟 | ~200 ms（轮询周期） | ~50 ms（IviumSoft 轮询） | ~50 ms（IviumSoft 轮询） |
| 是否可通过网络工作 | 是 | 否 | 否 |
| 信号是否可逆 | 是（再次设置） | 是（释放线） | 是（降低 DAC） |
| IviumSoft 批处理端 | `WaitForStatusPar` | `WaitForDigIn1` | `WaitForAn1` |
| 是否可接收信号 | 是（`get_status_par`） | 是（`get_digital_input`） | 是（`get_adc`） |

### 两端的超时都很重要

所有三个 IviumSoft `WaitFor...` DirectCommand 都接受**超时**设置。如果触发器始终未到达，批处理在超时后继续执行而不是永久挂起。始终配置：
- 足够长以覆盖最坏情况的 Python 准备时间
- 足够短以在出现问题时快速失败

将 Python 端 `wait_for_status()` 调用中的 `timeout_s` 与相同的时间预算匹配。

## 10. 完整触发工作流模板

一个独立示例，实现：
1. 重置所有实例
2. 同时向所有实例触发软件信号
3. 等待所有实例报告完成
4. 从所有实例收集数据

In [ ]:
import time
import pandas as pd
from pyvium import Pyvium

STATUS_IDLE    = 0
STATUS_GO      = 1
STATUS_RUNNING = 2
STATUS_DONE    = 3
STATUS_ABORT   = -99
TRIGGER_TIMEOUT_S = 120.0

Pyvium.open_driver()
instances = Pyvium.get_active_iviumsoft_instances()
print(f"设备: {instances}")

# 第 1 步：重置所有
broadcast_status(STATUS_IDLE)

# 第 2 步：连接所有
for inst in instances:
    Pyvium.select_iviumsoft_instance(inst)
    Pyvium.connect_device()
    print(f"  实例 {inst}: 已连接")

# 第 3 步：同时触发所有
# 每个实例上的 IviumSoft 批处理必须配置 WaitForStatusPar（value=1）
broadcast_status(STATUS_GO)

# 第 4 步：等待所有完成（IviumSoft 通过 SetStatusPar 将状态设为 3）
all_done = wait_all_status(instances, target=STATUS_DONE, timeout_s=TRIGGER_TIMEOUT_S)

if not all_done:
    broadcast_status(STATUS_ABORT)
    print("已中止 — 不是所有实例都在时间内完成")
else:
    # 第 5 步：从所有实例收集数据
    results = {}
    for inst in instances:
        Pyvium.select_iviumsoft_instance(inst)
        n = Pyvium.get_available_data_points_number()
        rows = [Pyvium.get_data_point(i) for i in range(1, n + 1)]
        results[inst] = pd.DataFrame(rows, columns=["E (V)", "I (A)", "aux"])
        print(f"  实例 {inst}: {len(rows)} 个点")

Pyvium.close_driver()
print("完成")

## 清理

In [ ]:
# 将所有数字和模拟输出重置为安全状态
Pyvium.set_digital_output(0)
Pyvium.set_dac(0, 0.0)
Pyvium.set_dac(1, 0.0)
Pyvium.close_driver()
print("输出已重置，驱动已关闭")

---

## 小结

### 状态参数

| 任务 | Python | IviumSoft 批处理 |
|------|--------|------------------|
| 重置 | `set_status_par(0)` | — |
| 告知 IviumSoft 开始 | `set_status_par(1)` | `WaitForStatusPar`（value=1，超时） |
| 等待 IviumSoft 完成 | 轮询循环中的 `get_status_par()` | `SetStatusPar`（value=3） |
| 广播到所有实例 | 循环 `select_iviumsoft_instance(n)` + `set_status_par(v)` | — |
| PC 启动时的初始值 | — | **-1**（保留直到 PC 重启） |

### 硬件触发器

| 类型 | Python 发送 | IviumSoft 批处理等待 | 读回 |
|------|-------------|--------------------------|------|
| 数字 | `set_digital_output(bitmask)` | `WaitForDigIn1`（高/低，超时） | `get_digital_input()` |
| 模拟 | `set_dac(0, voltage)` | `WaitForAn1 > threshold`（超时） | `get_adc(channel)` |

### IviumSoft 驱动 Python

| 方法 | IviumSoft 批处理 | Python 端 |
|--------|----------------|-------------|
| 启动脚本 | `ExecuteProgram` + `WaitUntilFinished` | 独立 `.py` 文件，以 `sys.exit(0)` 退出 |

### 安全检查清单

- 每次新工作流开始前始终将状态参数重置为 `0`
- 始终在 IviumSoft 端的 `WaitForStatusPar` / `WaitForDigIn1` / `WaitForAn1` 中配置超时
- 始终在 Python 轮询端设置匹配的 `timeout_s`
- 在清理单元格中将 DAC 重置为 `0 V`，将数字输出重置为 `0`
- 防御性调用 `abort_method()` 时捕获 `IllegalCommandError`

---

## 相关笔记本

| 笔记本 | 主题 |
|----------|-------|
| `04_direct_mode_signals.ipynb` | DAC 输出、ADC 输入、数字 I/O 参考 |
| `06_method_mode.ipynb` | 运行和监控方法，`abort_method()` |
| `08_batch_and_synchronization.ipynb` | 多设备设定点、并行测量、同步通道启动 |